# 1. Imports and functions

In [34]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [35]:
import os
import pandas as pd
from dotenv import load_dotenv
from matplotlib import pyplot as plt
from sklearn.metrics import f1_score, accuracy_score
from typing import List, Tuple

from wsd.load_data import load_data
from lineval.load_data import get_docs, docs_to_X_ys

from linpub.metrics import accuracy

from lineval.disambiguation_eval_backup import ClusterByMeaningModelv3
from lineval.disambiguation_eval_backup import OpenAiWordSenseComparatorv3, show_plots
from wsd.models import DummyComparator, ClusterByMeaningModel

from linalgo.hub.client import LinalgoClient

from etl.models import Candidate
from linalgo.annotate.models import Annotation

In [44]:
openai_api_key = os.getenv('OPEN_AI_API_KEY')
comp = OpenAiWordSenseComparatorv3(openai_api_key,
                                   openai_model='gpt-4o-mini',
                                   thought_process=False)
comp_4o = OpenAiWordSenseComparatorv3(openai_api_key,
                                      openai_model='gpt-4o',
                                      thought_process=False)
comp_4o_thought = OpenAiWordSenseComparatorv3(openai_api_key,
                                              openai_model='gpt-4o',
                                              thought_process=True)
comp_thought = OpenAiWordSenseComparatorv3(openai_api_key,
                                           openai_model='gpt-4o-mini',
                                           thought_process=True)
model1_1 = ClusterByMeaningModelv3(word_sense_comparator=comp)
model1_2 = ClusterByMeaningModelv3(word_sense_comparator=comp_4o)
model1_3 = ClusterByMeaningModelv3(word_sense_comparator=comp_4o_thought)
model1_4 = ClusterByMeaningModelv3(word_sense_comparator=comp_thought)

In [45]:
dummy2 = DummyComparator(probability=1)
model2 = ClusterByMeaningModel(comparator=dummy2)

dummy3 = DummyComparator(probability=0)
model3 = ClusterByMeaningModel(comparator=dummy3)

In [46]:
def reduce_X_y(X: List[Candidate], y: List[int]) -> Tuple[List[Candidate], List[int]]:
    """Reduce X and y to rows of 3 examples and 2 meanings per lemma-pos pair

    Note: this is only for lagre datasets, as it is very selective.
    ------
    Parameters:
    X: List[Candidate]
        List of candidates
    y: List[int]
        List of labels
    ------
    Returns:
    Tuple[List[Candidate], List[int]]
        Balanced X and y
    """
    X_df = pd.DataFrame(X)

    # the models that I coded work with context, not examples
    X_df["context"] = [c.example for c in X]
    X_df = X_df.drop(columns=["example"])

    X_df["y"] = y
    X_df_len3 = X_df.groupby(["lemma", "pos"]).filter(lambda x: len(x) == 3)
    X_df_bl = pd.DataFrame()
    for _, group in X_df_len3.groupby(["lemma", "pos"]):
        if group["y"].nunique() == 2:
            #sort group by number of similar meanings
            group["count"] = group["y"].map(group["y"].value_counts())
            group = group.sort_values(by="count")
            # order will now be b/a/a or a/b/b so we just permute the first two
            group = pd.concat([group.iloc[[1]], group.iloc[[0]], group.iloc[[2]]])
            # order will now be a/b/a or b/a/b
            X_df_bl = pd.concat([X_df_bl, group])
    bl_y = X_df_bl["y"].to_list()
    cand_df = X_df_bl.drop(columns=["y", "count"])
    bl_X = [Candidate(**row) for row in cand_df.to_dict(orient="records")]
    return bl_X, bl_y

In [47]:
def v3_load_or_predict(file: str,
                       model : ClusterByMeaningModelv3,
                       X: List[Candidate]
                       ) -> Tuple[List[Candidate], List[int], pd.DataFrame]:
    """Load predictions from file or predict them

    Note: This function only works with v3 models,
    Other models just have predict and cannot load and outuput preds and process

    ------
    Parameters:
    file: str
        File path
    model: ClusterByMeaningModel
        Model to predict with
    X: List[Candidate]
        List of candidates
    ------
    Returns:
    Tuple[List[Candidate], List[int], pd.DataFrame]
        X, for when loaded from file
        y_pred,
        DataFrame with predictions and process
        """
    if os.path.exists(file):
        df = pd.read_csv(file)
        y_pred = df["y_pred"].to_list()
        process_lst = df["process"].to_list()
        pred_lst = df["pred"].to_list()
        df.drop(columns=["y_pred","process","pred"], inplace=True)
        X = [Candidate(**row) for row in df.to_dict(orient="records")]
        df = pd.DataFrame(X)
        df["process"] = process_lst
        df["pred"] = pred_lst
    else:
        X, y_pred = model.predict(X, verbose=True)
        df = pd.DataFrame(X)
        df["y_pred"] = y_pred
        df["pred"] = [c.pred for c in X]
        process_list = []
        for c in X:
            if not hasattr(c, "process"):
                process_list.append("NA")
            else :
                process_list.append(c.process)
        df["process"] = process_list
        df.to_csv(file, index=False)
    return X, y_pred, df

In [48]:
def get_f1_score(df, y, y_pred):
    X_df = df
    X_df["y"] = y
    X_df["y_pred"] = y_pred
    f1_y = []
    f1_y_pred = []
    for _, group in X_df.groupby(["lemma", "pos"]):
        #sort group by number of similar meanings
        group["count"] = group["y"].map(group["y"].value_counts())
        group = group.sort_values(by="count")
        # order will now be b/a/a or a/b/b so we just permute the first two
        group = pd.concat([group.iloc[[1]], group.iloc[[0]], group.iloc[[2]]])
        # order will now be a/b/a or b/a/b
        pred1 = bool(group["y_pred"].iloc[1] == group["y_pred"].iloc[0])
        pred2 = bool(group["y_pred"].iloc[2] == group["y_pred"].iloc[0])
        f1_y.extend([False, True])
        f1_y_pred.extend([pred1, pred2])
    return f1_score(f1_y, f1_y_pred) , accuracy_score(f1_y, f1_y_pred)

def print_scores(df, y, y_pred1, y_pred2, y_pred3):
    print(f"--f1 scores--")
    print(f"Model: {get_f1_score(df, y, y_pred1)[0]}")
    print(f"Dummy(1): {get_f1_score(df, y, y_pred2)[0]}")
    print(f"Dummy(0): {get_f1_score(df, y, y_pred3)[0]}")
    print(f"--sklearn accuracy scores--")
    print(f"Model: {get_f1_score(df, y, y_pred1)[1]}")
    print(f"Dummy(1): {get_f1_score(df, y, y_pred2)[1]}")
    print(f"Dummy(0): {get_f1_score(df, y, y_pred3)[1]}")
    print(f"--linpub accuracy scores--")
    print(f"Model: {accuracy(y_pred1, y)}")
    print(f"Dummy(1): {accuracy(y_pred2, y)}")
    print(f"Dummy(0): {accuracy(y_pred3, y)}")

In [49]:
#need to add the correct column to the dataframe
def add_correct_column(df, y_pred1, bl_y):
    df["y"] = bl_y
    df["y_pred"] = y_pred1
    correct_df = pd.DataFrame()
    for _, gp in df.groupby(["lemma", "pos"]):
        #sort gp by number of similar meanings
        gp["count"] = gp["y"].map(gp["y"].value_counts())
        gp = gp.sort_values(by="count")
        # order will now be b/a/a or a/b/b so we just permute the first two
        gp = pd.concat([gp.iloc[[1]], gp.iloc[[0]], gp.iloc[[2]]])
        # order will now be a/b/a or b/a/b
        gp.drop(columns=["count"], inplace=True)
        pred1 = gp["y_pred"].iloc[1] != gp["y_pred"].iloc[0]
        pred2 = gp["y_pred"].iloc[2] == gp["y_pred"].iloc[0]
        gp = gp.copy()  # To avoid SettingWithCopyWarning
        gp["correct"] = pd.Series(["ref", pred1, pred2], index=gp.index)
        correct_df = pd.concat([correct_df, gp])
    return correct_df

# 2. massaged dataset, new metric

## 2.1 load

In [ ]:
X,y = load_data("fr")
for x in X :
    x.context = x.example
len(X), len(y)

## 2.1.2 Exploration

In [ ]:
# group by lemma and pos
X_df = pd.DataFrame(X)
X_df["y"] = y
cond = X_df.groupby(["lemma", "pos"]).size() <= 20# show only lemmas with more than 10 examples
X_df_20m = X_df.set_index(["lemma", "pos"]).loc[cond].reset_index()
X_df_20m.groupby(["lemma", "pos"]).size(). hist(bins=20)
plt.title("Number of examples in pos/lemma groups")

In [ ]:
# group by lemma and pos
cond = X_df.groupby(["lemma", "pos"]).size() > 20# show only lemmas with more than 10 examples
X_df_20p = X_df.set_index(["lemma", "pos"]).loc[cond].reset_index()
X_df_20p.groupby(["lemma", "pos"]).size().hist(bins=1020)
plt.title("Number of examples in pos/lemma groups")

In [ ]:
# Top pos lemma group by number of rows
X_df.groupby(["lemma", "pos"]).size().sort_values(ascending=False).head(10)

## 2.2 Dire (say) vs dire (tell)

In [ ]:
X_df[X_df["lemma"] == "dire"]["y"].value_counts()

In [ ]:
lone_cands = len(X_df.groupby(["lemma", "pos"]).filter(lambda x: len(x) == 1))
lone_cands, lone_cands/len(X_df)

In [ ]:
X_df[(X_df["lemma"] == "dire") & (X_df["y"] == "bn:00093287v\n")].head(20) #say

In [ ]:
X_df[(X_df["lemma"] == "dire") & (X_df["y"] != "bn:00093287v\n")].head(20) #tell

## 2.3 Creating a balanded set

In [ ]:
X,y = load_data("fr")
for x in X :
    x.context = x.example
bl_X, bl_y = reduce_X_y(X, y)
len(bl_X), len(bl_y)

## 2.4 Running

In [ ]:
X, y_pred1, df = v3_load_or_predict("results1_1.csv", model1_1, bl_X)

y_pred2 = model2.predict(bl_X, verbose=True)
y_pred3 = model3.predict(bl_X, verbose=True)

## 2.4.2 Metrics

In [ ]:
print_scores(df, bl_y, y_pred1, y_pred2, y_pred3)

## 2.5 Plotting

In [ ]:
correct_df = add_correct_column(df, y_pred1, bl_y)
correct_df.head(3)

In [ ]:
show_plots(correct_df, errors=True)

## 2.6 For all models 🤘🔥🎸💀⚡ : balanced data

In [ ]:
X,y = load_data("fr")
for x in X :
    x.context = x.example
bl_X, bl_y = reduce_X_y(X, y)
len(bl_X), len(bl_y)

In [ ]:
y_pred2 = model2.predict(bl_X, verbose=True)
y_pred3 = model3.predict(bl_X, verbose=True)

In [ ]:
models = {"1_1": model1_1, "1_2": model1_2, "1_3": model1_3, "1_4": model1_4}

for key,value in models.items():
    X, y_pred, df = v3_load_or_predict(f"results{key}.csv", value, bl_X)
    print_scores(df, bl_y, y_pred, y_pred2, y_pred3)
    correct_df = add_correct_column(df, y_pred, bl_y)
    show_plots(correct_df, errors=True)

## For all modlels : raw data

In [ ]:
X,y = load_data("fr")
for x in X :
    x.context = x.example
len(X), len(y)

In [ ]:
size = 200
raw_X, raw_y = X[:size], y[:size]
len(raw_X), len(raw_y)

In [ ]:
y_pred2 = model2.predict(raw_X, verbose=True)
y_pred3 = model3.predict(raw_X, verbose=True)

In [ ]:
models = {"1_1": model1_1, "1_2": model1_2, "1_3": model1_3, "1_4": model1_4}

for key,value in models.items():
    X, y_pred, df = v3_load_or_predict(f"results{key}_raw{size}.csv", value, raw_X)
    print(f"Model accuracy: {accuracy(y_pred, raw_y)}")
    print(f"Dummy(1) accuracy: {accuracy(y_pred2, raw_y)}")
    print(f"Dummy(0) accuracy: {accuracy(y_pred3, raw_y)}")

# 3. Linhub dataset

In [18]:
docs = get_docs()

Retrivieving task with id 85d7052e-b464-47a9-beca-dd8df8f8c632...
Retrieving annotators... (1 found)
Retrieving entities... (7 found)
Retrieving documents... (63 found)
Retrieving annotations... (144 found)


In [13]:
X, y_db, y_human = docs_to_X_ys(docs)
for x in X :
    x.example = x.context
len(X), len(y_db), len(y_human)

(144, 144, 144)

In [14]:
y_pred2 = model2.predict(X, verbose=True)
y_pred3 = model3.predict(X, verbose=True)

100%|██████████| 63/63 [00:00<00:00, 5139.58it/s]


In [15]:
models = {"1_1": model1_1, "1_2": model1_2, "1_3": model1_3, "1_4": model1_4}


for key,value in models.items():
    X, y_pred, df = v3_load_or_predict(f"hub_results{key}.csv", value, X)
    print(f"-----model{key}  accuracies-----")
    print(f"Model accuracy: {accuracy(y_pred, y_human)}")
    print(f"Dummy(1) accuracy: {accuracy(y_pred2, y_human)}")
    print(f"Dummy(0) accuracy: {accuracy(y_pred3, y_human)}")

-----model1_1  accuracies-----
Model accuracy: 0.6904761904761905
Dummy(1) accuracy: 0.47619047619047616
Dummy(0) accuracy: 0.42361111111111116
-----model1_2  accuracies-----
Model accuracy: 0.6559139784946236
Dummy(1) accuracy: 0.47619047619047616
Dummy(0) accuracy: 0.42361111111111116
-----model1_3  accuracies-----
Model accuracy: 0.693069306930693
Dummy(1) accuracy: 0.47619047619047616
Dummy(0) accuracy: 0.42361111111111116
-----model1_4  accuracies-----
Model accuracy: 0.7159090909090908
Dummy(1) accuracy: 0.47619047619047616
Dummy(0) accuracy: 0.42361111111111116


# 4. getting old data from linpub

In [149]:
from lineval.load_data_backup import get_docs as get_old_docs, add_context

In [150]:
docs = get_old_docs()

Retrivieving task with id d3ce7764-eb85-4999-b965-c028f539ee33...
Retrieving annotators... (4 found)
Retrieving entities... (7 found)
Retrieving documents... (63 found)
Retrieving annotations... (428 found)


In [151]:
def get_X_y(docs):
    X = []
    y = []
    for d in docs :
        for a in list(d.annotations) :
            context = add_context(a).context
            candidate = Candidate(  id = a.id,
                                    lang = "fr",
                                    lemma = a.document.uri,
                                    pos = a.document.uri,
                                    context = context,
                                    example = context,
                                    lemma_meaning= a.entity.id,
                                    annotation= a.annotator.id,
            )
            X.append(candidate)
            y.append(hash((candidate.lemma_meaning, candidate.pos, candidate.lemma)))
    return X, y

def filter_annotations(X, y, annotator_id):
    X_filtered = []
    y_filtered = []
    for x_, y_ in zip(X, y):
        if x_.annotation == annotator_id:
            X_filtered.append(x_)
            y_filtered.append(y_)
    return X_filtered, y_filtered

abi_id = "eb9a2239-5a6c-4da5-8957-af8a90d3e880"
jack_id = "d34602e1-1664-42cb-b139-c5cb8bcfa2a0"
arnaud_id = "e920598f-774d-4e1c-a2dc-8fb9a6f3053f"

X, y = get_X_y(docs)

X_ab, y_ab = filter_annotations(X, y, abi_id)
X_j, y_j = filter_annotations(X, y, jack_id)
X_ar, y_ar = filter_annotations(X, y, arnaud_id)

len(X_ab), len(y_ab), len(X_j), len(y_j), len(y_ar), len(X_ar)

#Careful, the number of annotations and their order is different
#Therefore, the X is also different, so y_pred needs to be computed again
#Only metrics should be used

(141, 141, 141, 141, 142, 142)

In [152]:
X_ab, y_pred_ab, df_ab = v3_load_or_predict("hub_results1_3_abi.csv", model1_3, X_ab)
X_j, y_pred_j, df_j = v3_load_or_predict("hub_results1_3_jack.csv", model1_3, X_j)
X_ar, y_pred_ar, df_ar = v3_load_or_predict("hub_results1_3_arnaud.csv", model1_3, X_ar)
len(X_ab), len(y_pred_ab),len(df_ab), len(X_j), len(y_pred_j),len(df_j), len(X_ar), len(y_pred_ar),len(df_ar)

(141, 141, 141, 141, 141, 141, 142, 142, 142)

In [153]:
print(f"----- accurcies-----")
print(f"Abi truth, model accuracy: {accuracy(y_pred_ab, y_ab)}")
print(f"Jack truth, model accuracy: {accuracy(y_pred_j, y_j)}")
print(f"Arnaud truth, model accuracy: {accuracy(y_pred_ar, y_ar)}")

----- accurcies-----
Abi truth, model accuracy: 0.7777777777777778
Jack truth, model accuracy: 0.7070707070707071
Arnaud truth, model accuracy: 0.6363636363636364


In [136]:
df = pd.DataFrame(X)
df["y"] = y
df.head(3)

,id,lang,pos,text,text_meaning,text_romaji,text_hiragana,lemma,lemma_meaning,lemma_romaji,lemma_hiragana,example,context,annotation,document,status,y
0,7aa23f5e-5d78-4fa2-abf0-8d3004129455,fr,62,NA,NA,NA,NA,62,1ecf29a6-9193-4a8b-9317-5dc319b94507,NA,NA,"L'armée était en difficulté, il était préférab...","L'armée était en difficulté, il était préférab...",eb9a2239-5a6c-4da5-8957-af8a90d3e880,NA,pending,-658350140196628298
1,f55b699b-1ccd-4d67-acd9-c1f2435b24be,fr,62,NA,NA,NA,NA,62,371a5d39-6e41-427f-8e1f-6bf64504e080,NA,NA,"J’ai eu si mal au ventre, tout d’un coup, que ...","J’ai eu si mal au ventre, tout d’un coup, que ...",eb9a2239-5a6c-4da5-8957-af8a90d3e880,NA,pending,-1814635245826258559
2,0e775bd1-9946-4cdf-a2cc-4ec14b6b229a,fr,62,NA,NA,NA,NA,62,371a5d39-6e41-427f-8e1f-6bf64504e080,NA,NA,"L'armée était en difficulté, il était préférab...","L'armée était en difficulté, il était préférab...",e920598f-774d-4e1c-a2dc-8fb9a6f3053f,NA,pending,-1814635245826258559


In [137]:
df.groupby(["lemma", "example"]).size().value_counts()

3    118
6      7
2      4
9      1
4      1
7      1
Name: count, dtype: int64

In [139]:
massaged_df = pd.DataFrame()
for _, gp in df.groupby(["lemma", "example"]):
    if len(set(gp["annotation"])) == 3:
        ab_ano = gp[gp["annotation"] == abi_id].iloc[0]
        j_ano = gp[gp["annotation"] == jack_id].iloc[0]
        ar_ano = gp[gp["annotation"] == arnaud_id].iloc[0]
        filtered_gp = pd.DataFrame([ab_ano, j_ano, ar_ano])
        massaged_df = pd.concat([massaged_df, filtered_gp])
len(massaged_df)

381

In [141]:
massaged_y = massaged_df["y"].to_list()
massaged_df.drop(columns=["y"], inplace=True)
massaged_X = [Candidate(**row) for row in massaged_df.to_dict(orient="records")]
len(massaged_X), len(massaged_y)

(381, 381)

In [142]:
X_ab, y_ab = filter_annotations(massaged_X, massaged_y, abi_id)
X_j, y_j = filter_annotations(massaged_X, massaged_y, jack_id)
X_ar, y_ar = filter_annotations(massaged_X, massaged_y, arnaud_id)
len(X_ab), len(y_ab), len(X_j), len(y_j), len(X_ar), len(y_ar)

(127, 127, 127, 127, 127, 127)

In [145]:
X_ab[100],X_j[100],X_ar[100]

(Candidate(id='79dd9c37-1a9a-4957-9076-bc30db86eee1', lang='fr', pos='54', text='NA', text_meaning='NA', text_romaji='NA', text_hiragana='NA', lemma='54', lemma_meaning='1ecf29a6-9193-4a8b-9317-5dc319b94507', lemma_romaji='NA', lemma_hiragana='NA', example='Je comprends le français mais je ne le parle pas couramment.', context='Je comprends le français mais je ne le parle pas couramment.', annotation='eb9a2239-5a6c-4da5-8957-af8a90d3e880', document='NA', status='pending'),
 Candidate(id='f61e9702-6f35-4897-bdb7-2c2a9f79503c', lang='fr', pos='54', text='NA', text_meaning='NA', text_romaji='NA', text_hiragana='NA', lemma='54', lemma_meaning='1ecf29a6-9193-4a8b-9317-5dc319b94507', lemma_romaji='NA', lemma_hiragana='NA', example='Je comprends le français mais je ne le parle pas couramment.', context='Je comprends le français mais je ne le parle pas couramment.', annotation='d34602e1-1664-42cb-b139-c5cb8bcfa2a0', document='NA', status='pending'),
 Candidate(id='81d433d1-de42-4a14-ae34-60c09

In [148]:
print(f"Abi truth, Jack accuracy : {accuracy(y_ab, y_j)}")
print(f"Abi truth, Arnaud accuracy : {accuracy(y_ab, y_ar)}")
print(f"Jack truth, Abi accuracy : {accuracy(y_j, y_ab)}")
print(f"Jack truth, Arnaud accuracy : {accuracy(y_j, y_ar)}")
print(f"Arnaud truth, Abi accuracy : {accuracy(y_ar, y_ab)}")
print(f"Arnaud truth, Jack accuracy : {accuracy(y_ar, y_j)}")

Abi truth, Jack accuracy : 0.8351648351648352
Abi truth, Arnaud accuracy : 0.7912087912087912
Jack truth, Abi accuracy : 0.8636363636363636
Jack truth, Arnaud accuracy : 0.7840909090909091
Arnaud truth, Abi accuracy : 0.8674698795180723
Arnaud truth, Jack accuracy : 0.8313253012048193



-------
1. base mdel and data

1.1 The dummy 1 is better that the model at 91% vs 90%

1.2 The dataset is unbalanced

1.3 The model errors seem to be due to errors in the y

-------
2. balancing the dataset

2.1. There are 1433 dual candidates, ~6% of the dataset

2.2 The top pos/lemma group is dire, with 1020 rows (2 different meanings)(944of the main maining, and then 74 of the 2nd) (say vs tell)

2.3 From the pos/lemma groups with 3 rows and 2 meanings, (210 rows) I made a balanced, ordered dataset where the model will have to do 2 outputs: 0 then 1 

2.4. The model is barely better than the baseline